# w08 — Strengthening the capstone with real warehouse-scale data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Seif-2/ML-week-1-FLY/blob/main/work/notebooks/w08_warehouse_scale_model.ipynb?flush_cache=true)

Everything in w01–w07 and the capstone paper was built on the 30,000-row starter CSV. This notebook re-runs the same baseline-vs-model comparison on real warehouse data — two mid-panel months (never the `_sample`/final month, per the assignment's leakage warning) — to check whether the starter-slice findings hold at real scale.

**Run this in Colab** with your `HF_TOKEN` already saved in the Secrets panel (🔑 icon) — Runtime → Run all, then save back to GitHub, or paste the outputs back to Claude to finalize.

In [ ]:
# 0. Setup — installs + Hugging Face auth (token stays in Colab Secrets, never in a cell)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "scikit-learn", "pandas"], check=True)

import duckdb

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    import os, getpass
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF_TOKEN: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REPO_ID = "FlyRank/internship-warehouse"
# Two mid-panel months, deliberately not adjacent, not the final/_sample month
MONTHS = ["2026-02", "2026-04"]

print("DuckDB + Hugging Face auth ready. Months:", MONTHS)

In [ ]:
# Confirm real file paths and real column names before writing any query — same discipline as w03
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset", token=HF_TOKEN)

fact_relations = []
for m in MONTHS:
    matches = [f for f in files if "fact_content_daily_performance" in f and f"month={m}" in f]
    print(f"month={m}: {len(matches)} file(s) found")
    fact_relations.append(f"read_parquet('hf://datasets/{REPO_ID}/fact_content_daily_performance/month={m}/*.parquet')")

FACT_UNION = " UNION ALL ".join(f"SELECT * FROM {r}" for r in fact_relations)

print("\nSchema (first month's partition):")
con.sql(f"DESCRIBE SELECT * FROM {fact_relations[0]} LIMIT 0").show(max_rows=100)

In [ ]:
# Aggregate both months to content-client-month grain, same construction as w03/w05:
# gsc_impressions / gsc_clicks (real column names confirmed above), a volume floor,
# and the honest, non-CTR feature set.
agg_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        month,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_month,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_month,
        BOOL_OR(ga4_data_available IS TRUE) AS ga4_data_available_month
    FROM ({FACT_UNION})
    GROUP BY content_hash_id, client_hash_id, month
    HAVING SUM(gsc_impressions) >= 100
"""
feat = con.sql(agg_query).df()
feat["ctr_month"] = 100 * feat["clicks_month"] / feat["impressions_month"]

print(f"Warehouse feature frame: {feat.shape[0]:,} rows across {feat['month'].nunique()} months "
      f"and {feat['client_hash_id'].nunique():,} clients")
print(f"(compare: starter CSV had 16,726 visible rows across 32 clients, ONE 90-day window)")
feat.head(5)

In [ ]:
# Position tiers + label, same construction as every prior week
import numpy as np

def position_tier(p):
    if p <= 3: return "top_3"
    if p <= 10: return "page_1"
    if p <= 20: return "striking"
    return "deep"

feat = feat.dropna(subset=["avg_position_month"]).copy()
feat["position_tier"] = feat["avg_position_month"].apply(position_tier)
tier_median = feat.groupby("position_tier")["ctr_month"].transform("median")
feat["tier_median_ctr"] = tier_median
feat["ctr_gap"] = tier_median - feat["ctr_month"]
feat["underperform_flag"] = (feat["ctr_gap"] > 0).astype(int)
feat["baseline_score"] = feat["ctr_gap"] * feat["impressions_month"]

print(f"Base rate (underperform_flag=1): {feat['underperform_flag'].mean():.3f}")
print(feat.groupby("position_tier").agg(n=("ctr_month","size"), median_ctr=("ctr_month","median")).sort_values("median_ctr", ascending=False))

In [ ]:
# Grouped-by-client split (same discipline as w05/w06), train + compare baseline vs LogReg vs RF
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import pandas as pd

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(feat, groups=feat["client_hash_id"]))
train, test = feat.iloc[tr_idx].copy(), feat.iloc[te_idx].copy()
overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"Train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows, {test['client_hash_id'].nunique()} clients")
print(f"Client overlap: {len(overlap)} (0 = clean)\n")

K_VALUES = (50, 200)
results = []

base_rate = test["underperform_flag"].mean()
baseline_auc = roc_auc_score(test["underperform_flag"], test["baseline_score"])
row = {"method": "Baseline (ctr_gap x impressions)", "roc_auc": baseline_auc}
for k in K_VALUES:
    row[f"precision@{k}"] = precision_at_k(test["baseline_score"], test["underperform_flag"].values, k)
results.append(row)

num_feats = ["impressions_month", "avg_position_month", "days_with_impressions_month"]
cat_feats = ["position_tier"]
Xtr = pd.get_dummies(train[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte = pd.get_dummies(test[num_feats + cat_feats], columns=cat_feats).fillna(0)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
ytr, yte = train["underperform_flag"], test["underperform_flag"]

scaler = StandardScaler()
Xtr_s, Xte_s = scaler.fit_transform(Xtr), scaler.transform(Xte)
lr = LogisticRegression(max_iter=2000, random_state=42).fit(Xtr_s, ytr)
lr_scores = lr.predict_proba(Xte_s)[:, 1]
row = {"method": "Logistic Regression (honest features)", "roc_auc": roc_auc_score(yte, lr_scores)}
for k in K_VALUES:
    row[f"precision@{k}"] = precision_at_k(lr_scores, yte.values, k)
results.append(row)

rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
rf_scores = rf.predict_proba(Xte)[:, 1]
row = {"method": "Random Forest (honest features)", "roc_auc": roc_auc_score(yte, rf_scores)}
for k in K_VALUES:
    row[f"precision@{k}"] = precision_at_k(rf_scores, yte.values, k)
results.append(row)

results_df = pd.DataFrame(results).round(3)
print(f"Base rate on TEST: {base_rate:.3f}\n")
results_df

In [ ]:
# Side-by-side: does the starter-CSV finding hold at warehouse scale?
comparison = pd.DataFrame([
    {"metric": "n (test rows)", "starter_csv_w05": 1461, "warehouse_2months": len(test)},
    {"metric": "base_rate", "starter_csv_w05": 0.455, "warehouse_2months": round(base_rate, 3)},
    {"metric": "RF ROC AUC", "starter_csv_w05": 0.658, "warehouse_2months": round(results_df.loc[2,'roc_auc'], 3)},
    {"metric": "RF Precision@200", "starter_csv_w05": 0.550, "warehouse_2months": round(results_df.loc[2,'precision@200'], 3)},
])
print(comparison.to_string(index=False))
print("\nIf warehouse numbers land close to the starter-CSV numbers, that's real evidence the pattern")
print("generalizes beyond the small starter slice, not just an artifact of a small sample.")

In [ ]:
# Export the metrics receipt for the paper
import json, os
os.makedirs("work/outputs", exist_ok=True)

warehouse_metrics = {
    "months_used": MONTHS,
    "n_test": int(len(test)),
    "n_clients_test": int(test["client_hash_id"].nunique()),
    "client_overlap": int(len(overlap)),
    "base_rate": float(base_rate),
    "results_table": results_df.to_dict(orient="records"),
}
with open("work/outputs/w08_warehouse_metrics.json", "w") as f:
    json.dump(warehouse_metrics, f, indent=2)
print("Saved: work/outputs/w08_warehouse_metrics.json")
print(json.dumps(warehouse_metrics, indent=2))